**Data Import and Basic Operation**

In [111]:
import pandas as pd
import requests
import time
df = pd.read_csv("/content/city_day.csv")
print(df.shape)
df.head()
df = df.rename(columns={"PM2.5": "PM2_5"})
df.info()

(21128, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21128 entries, 0 to 21127
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   City    21128 non-null  object 
 1   Date    21128 non-null  object 
 2   AQI     21128 non-null  float64
 3   PM10    21128 non-null  float64
 4   PM2_5   21128 non-null  float64
 5   NO2     21128 non-null  float64
 6   NO      21128 non-null  float64
 7   SO2     21128 non-null  float64
 8   CO      21128 non-null  float64
 9   O3      21128 non-null  float64
dtypes: float64(8), object(2)
memory usage: 1.6+ MB


**Data Engineering 1**

In [112]:
df = df.copy()

# Make sure we have a single, consistent Date column
if "date" in df.columns:
    df.rename(columns={"date": "Date"}, inplace=True)

df["Date"] = pd.to_datetime(df["Date"]).dt.date
df["City"] = df["City"].astype(str).str.strip()



In [113]:
#In this code block we geocode the cities to later find out their Geographical Information!!!
def geocode_city_openmeteo(city, country_code="IN"):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": city, "count": 1, "language": "en", "format": "json", "country_code": country_code}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    if "results" not in js or len(js["results"]) == 0:
        return None
    best = js["results"][0]
    return {"City": city, "latitude": best["latitude"], "longitude": best["longitude"]}

cities = sorted(df["City"].unique())

geo_rows, failed_geo = [], []
for c in cities:
    g = geocode_city_openmeteo(c, country_code="IN")
    if g is None:
        failed_geo.append(c)
    else:
        geo_rows.append(g)
    time.sleep(0.3)

geo_df = pd.DataFrame(geo_rows)
print("Geocoded:", len(geo_df), "Failed:", len(failed_geo))
print("Failed:", failed_geo)

Geocoded: 21 Failed: 1
Failed: ['Brajrajnagar']


**Geographical Features Access**

In [114]:
session = requests.Session()

def fetch_openmeteo_daily_with_retry(lat, lon, start_date, end_date, timezone="Asia/Kolkata",
                                     max_retries=8, base_sleep=2.0):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "relative_humidity_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ],
        "timezone": timezone
    }

    for attempt in range(max_retries):
        r = session.get(url, params=params, timeout=60)

        if r.status_code == 200:
            js = r.json()
            w = pd.DataFrame(js["daily"]).rename(columns={"time": "Date"})
            w["Date"] = pd.to_datetime(w["Date"]).dt.date
            return w

        if r.status_code == 429:
            sleep_s = min(base_sleep * (2 ** attempt), 60)
            print(f"429 rate limit. Sleeping {sleep_s:.1f}s...")
            time.sleep(sleep_s)
            continue

        r.raise_for_status()

    raise RuntimeError("Failed after retries")

weather_all = []

for _, row in geo_df.iterrows():
    c = row["City"]
    lat, lon = row["latitude"], row["longitude"]

    df_c = df[df["City"] == c]
    start_date = str(df_c["Date"].min())
    end_date   = str(df_c["Date"].max())

    try:
        w = fetch_openmeteo_daily_with_retry(lat, lon, start_date, end_date)
        w["City"] = c
        weather_all.append(w)
        print("✅ Weather:", c)
    except Exception as e:
        print("❌ Weather failed:", c, "->", e)

    time.sleep(1.0)

weather_df = pd.concat(weather_all, ignore_index=True)
print("weather_df shape:", weather_df.shape)
weather_df.head()


✅ Weather: Ahmedabad
✅ Weather: Aizawl
✅ Weather: Amaravati
✅ Weather: Amritsar
✅ Weather: Bengaluru
✅ Weather: Bhopal
✅ Weather: Chandigarh
✅ Weather: Chennai
✅ Weather: Delhi
✅ Weather: Gurugram
✅ Weather: Guwahati
✅ Weather: Hyderabad
✅ Weather: Jaipur
✅ Weather: Jorapokhar
✅ Weather: Kolkata
✅ Weather: Lucknow
429 rate limit. Sleeping 2.0s...
429 rate limit. Sleeping 4.0s...
429 rate limit. Sleeping 8.0s...
429 rate limit. Sleeping 16.0s...
✅ Weather: Mumbai
✅ Weather: Patna
✅ Weather: Shillong
✅ Weather: Talcher
✅ Weather: Thiruvananthapuram
weather_df shape: (22587, 7)


,Date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,City
0,2015-02-05,29.1,16.7,44,0.0,15.5,Ahmedabad
1,2015-02-06,29.9,16.7,52,0.0,11.0,Ahmedabad
2,2015-02-07,29.5,16.2,37,0.0,10.7,Ahmedabad
3,2015-02-08,28.7,14.6,37,0.0,10.7,Ahmedabad
4,2015-02-09,29.0,14.7,46,0.0,13.7,Ahmedabad


In [115]:
#The code below merges both the csv and the weather features together.
df2 = df.merge(weather_df, on=["City", "Date"], how="left")

weather_cols = [
    "temperature_2m_max","temperature_2m_min",
    "relative_humidity_2m_mean","precipitation_sum",
    "wind_speed_10m_max"
]

print("Missing weather ratio:")
print(df2[weather_cols].isna().mean())


Missing weather ratio:
temperature_2m_max           0.031427
temperature_2m_min           0.031427
relative_humidity_2m_mean    0.031427
precipitation_sum            0.031427
wind_speed_10m_max           0.031427
dtype: float64


In [116]:
weather_df.to_csv("weather_cache.csv", index=False)
df2.to_csv("aqi_with_weather.csv", index=False)
print("✅ Saved weather_cache.csv and aqi_with_weather.csv")

✅ Saved weather_cache.csv and aqi_with_weather.csv


In [117]:
df = df.copy()
df["date"] = pd.to_datetime(df["Date"]).dt.date

In [118]:
weather_all = []
failed_weather = []

for _, row in geo_df.iterrows():

    c = row["City"]
    lat, lon = row["latitude"], row["longitude"]

    df_c = df[df["City"] == c]

    start_date = str(df_c["Date"].min())
    end_date   = str(df_c["Date"].max())

    try:
        w = fetch_openmeteo_daily_with_retry(lat, lon, start_date, end_date)
        w["City"] = c
        weather_all.append(w)
        print("✅ Weather:", c)

    except Exception as e:
        failed_weather.append(c)
        print("❌ Weather failed:", c)

    time.sleep(1.0)   # important to avoid 429

weather_df = pd.concat(weather_all, ignore_index=True)

print("Weather failures:", failed_weather)


✅ Weather: Ahmedabad
✅ Weather: Aizawl
✅ Weather: Amaravati
✅ Weather: Amritsar
✅ Weather: Bengaluru
✅ Weather: Bhopal
✅ Weather: Chandigarh
✅ Weather: Chennai
✅ Weather: Delhi
✅ Weather: Gurugram
✅ Weather: Guwahati
✅ Weather: Hyderabad
429 rate limit. Sleeping 2.0s...
429 rate limit. Sleeping 4.0s...
429 rate limit. Sleeping 8.0s...
429 rate limit. Sleeping 16.0s...
429 rate limit. Sleeping 32.0s...
✅ Weather: Jaipur
✅ Weather: Jorapokhar
✅ Weather: Kolkata
✅ Weather: Lucknow
✅ Weather: Mumbai
✅ Weather: Patna
✅ Weather: Shillong
✅ Weather: Talcher
✅ Weather: Thiruvananthapuram
Weather failures: []


In [119]:
df2 = df.merge(weather_df, on=["City", "Date"], how="left")

weather_cols = [
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "precipitation_sum",
    "wind_speed_10m_max"
]

print("Missing weather ratio:")
print(df2[weather_cols].isna().mean())


Missing weather ratio:
temperature_2m_max           0.031427
temperature_2m_min           0.031427
relative_humidity_2m_mean    0.031427
precipitation_sum            0.031427
wind_speed_10m_max           0.031427
dtype: float64


In [120]:
print(weather_df.head())
print(weather_df.dtypes)


         Date  temperature_2m_max  temperature_2m_min  \
0  2015-02-05                29.1                16.7   
1  2015-02-06                29.9                16.7   
2  2015-02-07                29.5                16.2   
3  2015-02-08                28.7                14.6   
4  2015-02-09                29.0                14.7   

   relative_humidity_2m_mean  precipitation_sum  wind_speed_10m_max       City  
0                         44                0.0                15.5  Ahmedabad  
1                         52                0.0                11.0  Ahmedabad  
2                         37                0.0                10.7  Ahmedabad  
3                         37                0.0                10.7  Ahmedabad  
4                         46                0.0                13.7  Ahmedabad  
Date                          object
temperature_2m_max           float64
temperature_2m_min           float64
relative_humidity_2m_mean      int64
precipitation_sum      

In [121]:
print(df.head())
print(df.dtypes)


        City        Date    AQI    PM10   PM2_5    NO2     NO    SO2    CO  \
0  Ahmedabad  2015-02-05  149.0  122.41   58.36  21.39  10.05  32.66  2.60   
1  Ahmedabad  2015-02-06  190.0  122.41   79.29  26.94  10.05  67.41  1.16   
2  Ahmedabad  2015-02-07  247.0  122.41   88.70  31.32  10.05  80.09  7.29   
3  Ahmedabad  2015-02-08  379.0  122.41   74.28  27.30  10.05  54.28  8.92   
4  Ahmedabad  2015-02-09  341.0  122.41  113.93  24.27  10.05  48.73  4.32   

      O3        date  
0  53.54  2015-02-05  
1  59.30  2015-02-06  
2  44.76  2015-02-07  
3  47.42  2015-02-08  
4  39.94  2015-02-09  
City      object
Date      object
AQI      float64
PM10     float64
PM2_5    float64
NO2      float64
NO       float64
SO2      float64
CO       float64
O3       float64
date      object
dtype: object


In [122]:
df2.info()
df2.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21128 entries, 0 to 21127
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   City                       21128 non-null  object 
 1   Date                       21128 non-null  object 
 2   AQI                        21128 non-null  float64
 3   PM10                       21128 non-null  float64
 4   PM2_5                      21128 non-null  float64
 5   NO2                        21128 non-null  float64
 6   NO                         21128 non-null  float64
 7   SO2                        21128 non-null  float64
 8   CO                         21128 non-null  float64
 9   O3                         21128 non-null  float64
 10  date                       21128 non-null  object 
 11  temperature_2m_max         20464 non-null  float64
 12  temperature_2m_min         20464 non-null  float64
 13  relative_humidity_2m_mean  20464 non-null  flo

,0
City,0
Date,0
AQI,0
PM10,0
PM2_5,0
NO2,0
NO,0
SO2,0
CO,0
O3,0


**Data Engineering 2**

In [124]:
#We drop columns which have less impact on AQI(and on top of that they have missing values(a lot))
#df2.drop(columns=['Xylene'], inplace=True)

In [125]:
df2.isna().sum()

,0
City,0
Date,0
AQI,0
PM10,0
PM2_5,0
NO2,0
NO,0
SO2,0
CO,0
O3,0


In [126]:
#In this code we use a method(in this case interpolation) for filling the missing values in the column

cols = ['PM2_5', 'PM10', 'NO', 'NO2',
        'CO', 'SO2', 'O3']

for col in cols:
    df2[col] = df2.groupby('City')[col].transform(
        lambda x: x.interpolate(method='linear')
    )

In [127]:
df2.isna().sum()

,0
City,0
Date,0
AQI,0
PM10,0
PM2_5,0
NO2,0
NO,0
SO2,0
CO,0
O3,0


In [128]:
df2.head()

,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max
0,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54,2015-02-05,29.1,16.7,44.0,0.0,15.5
1,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30,2015-02-06,29.9,16.7,52.0,0.0,11.0
2,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76,2015-02-07,29.5,16.2,37.0,0.0,10.7
3,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42,2015-02-08,28.7,14.6,37.0,0.0,10.7
4,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94,2015-02-09,29.0,14.7,46.0,0.0,13.7


In [129]:
df2 = df2.drop(columns = ['AQI_Bucket','Toluene', 'NH3'], errors='ignore')

In [130]:
df2.head()

,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max
0,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54,2015-02-05,29.1,16.7,44.0,0.0,15.5
1,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30,2015-02-06,29.9,16.7,52.0,0.0,11.0
2,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76,2015-02-07,29.5,16.2,37.0,0.0,10.7
3,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42,2015-02-08,28.7,14.6,37.0,0.0,10.7
4,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94,2015-02-09,29.0,14.7,46.0,0.0,13.7


In [131]:
#It converts the Date column to datetime objects
df2['Date'] = pd.to_datetime(df2['Date'])
df2 = df2.sort_values(['City','Date'])

In [132]:
df2.head()

,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max
0,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54,2015-02-05,29.1,16.7,44.0,0.0,15.5
1,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30,2015-02-06,29.9,16.7,52.0,0.0,11.0
2,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76,2015-02-07,29.5,16.2,37.0,0.0,10.7
3,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42,2015-02-08,28.7,14.6,37.0,0.0,10.7
4,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94,2015-02-09,29.0,14.7,46.0,0.0,13.7


In [133]:
#This block helps us in learning the trends.
df2 = df2.sort_values(['City', 'Date'])

df2['AQI_rolling7'] = (
    df2.groupby('City')['AQI']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

df2['AQI_change'] = df2['AQI'] - df2['AQI_rolling7']


In [134]:
df2.head()

,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change
0,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54,2015-02-05,29.1,16.7,44.0,0.0,15.5,149.000000,0.000000
1,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30,2015-02-06,29.9,16.7,52.0,0.0,11.0,169.500000,20.500000
2,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76,2015-02-07,29.5,16.2,37.0,0.0,10.7,195.333333,51.666667
3,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42,2015-02-08,28.7,14.6,37.0,0.0,10.7,241.250000,137.750000
4,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94,2015-02-09,29.0,14.7,46.0,0.0,13.7,261.200000,79.800000


In [135]:
#In this we use one more method to fill the empty spaces.
weather_cols = [
    "temperature_2m_max","temperature_2m_min",
    "relative_humidity_2m_mean","precipitation_sum",
    "wind_speed_10m_max"
]

# forward/back fill per city, then global median as last fallback
df2[weather_cols] = (
    df2.groupby("City")[weather_cols]
       .transform(lambda x: x.ffill().bfill())
)

df2[weather_cols] = df2[weather_cols].fillna(df2[weather_cols].median(numeric_only=True))

print("Missing weather ratio after fill:")
print(df2[weather_cols].isna().mean())


Missing weather ratio after fill:
temperature_2m_max           0.0
temperature_2m_min           0.0
relative_humidity_2m_mean    0.0
precipitation_sum            0.0
wind_speed_10m_max           0.0
dtype: float64


In [136]:
df2 = df2.dropna(subset=['AQI'])

In [137]:
features = [
    'PM10','PM2_5','NO2','NO','SO2','CO','O3',
    'construction_intensity',
    'temperature_2m_max','temperature_2m_min',
    'relative_humidity_2m_mean','precipitation_sum',
    'wind_speed_10m_max'
]


In [138]:
df2.head()

,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change
0,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54,2015-02-05,29.1,16.7,44.0,0.0,15.5,149.000000,0.000000
1,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30,2015-02-06,29.9,16.7,52.0,0.0,11.0,169.500000,20.500000
2,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76,2015-02-07,29.5,16.2,37.0,0.0,10.7,195.333333,51.666667
3,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42,2015-02-08,28.7,14.6,37.0,0.0,10.7,241.250000,137.750000
4,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94,2015-02-09,29.0,14.7,46.0,0.0,13.7,261.200000,79.800000


In [139]:
for col in cols:
    df2[col] = df2.groupby('City')[col].transform(
        lambda x: x.ffill().bfill()
    )


In [140]:
df2[cols].isna().sum()

,0
PM2_5,0
PM10,0
NO,0
NO2,0
CO,0
SO2,0
O3,0


In [141]:
df2.head()

,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change
0,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54,2015-02-05,29.1,16.7,44.0,0.0,15.5,149.000000,0.000000
1,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30,2015-02-06,29.9,16.7,52.0,0.0,11.0,169.500000,20.500000
2,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76,2015-02-07,29.5,16.2,37.0,0.0,10.7,195.333333,51.666667
3,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42,2015-02-08,28.7,14.6,37.0,0.0,10.7,241.250000,137.750000
4,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94,2015-02-09,29.0,14.7,46.0,0.0,13.7,261.200000,79.800000


In [142]:
df2.head()

,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change
0,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54,2015-02-05,29.1,16.7,44.0,0.0,15.5,149.000000,0.000000
1,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30,2015-02-06,29.9,16.7,52.0,0.0,11.0,169.500000,20.500000
2,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76,2015-02-07,29.5,16.2,37.0,0.0,10.7,195.333333,51.666667
3,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42,2015-02-08,28.7,14.6,37.0,0.0,10.7,241.250000,137.750000
4,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94,2015-02-09,29.0,14.7,46.0,0.0,13.7,261.200000,79.800000


In [143]:
df2.isna().sum()

,0
City,0
Date,0
AQI,0
PM10,0
PM2_5,0
NO2,0
NO,0
SO2,0
CO,0
O3,0


In [144]:
df2 = df2.dropna(subset=['AQI_rolling7', 'AQI_change'])

In [145]:
df2 = df2.copy()

In [146]:
for col in ['PM10', 'NO']:
    df2[col] = df2[col].fillna(df2[col].median())

In [147]:
df2.head()

,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change
0,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54,2015-02-05,29.1,16.7,44.0,0.0,15.5,149.000000,0.000000
1,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30,2015-02-06,29.9,16.7,52.0,0.0,11.0,169.500000,20.500000
2,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76,2015-02-07,29.5,16.2,37.0,0.0,10.7,195.333333,51.666667
3,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42,2015-02-08,28.7,14.6,37.0,0.0,10.7,241.250000,137.750000
4,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94,2015-02-09,29.0,14.7,46.0,0.0,13.7,261.200000,79.800000


In [148]:
df2.isna().sum()

,0
City,0
Date,0
AQI,0
PM10,0
PM2_5,0
NO2,0
NO,0
SO2,0
CO,0
O3,0


In [149]:
df2[['City','Date','AQI','AQI_rolling7','AQI_change']].head(20)


,City,Date,AQI,AQI_rolling7,AQI_change
0,Ahmedabad,2015-02-05,149.0,149.000000,0.000000
1,Ahmedabad,2015-02-06,190.0,169.500000,20.500000
2,Ahmedabad,2015-02-07,247.0,195.333333,51.666667
3,Ahmedabad,2015-02-08,379.0,241.250000,137.750000
4,Ahmedabad,2015-02-09,341.0,261.200000,79.800000
5,Ahmedabad,2015-02-10,256.0,260.333333,-4.333333
6,Ahmedabad,2015-02-11,388.0,278.571429,109.428571
7,Ahmedabad,2015-02-12,288.0,298.428571,-10.428571
8,Ahmedabad,2015-02-13,510.0,344.142857,165.857143
9,Ahmedabad,2015-02-14,761.0,417.571429,343.428571


In [150]:
df2[df2['City'] == 'Ahmedabad'][['Date','AQI','AQI_rolling7','AQI_change']].head(20)


,Date,AQI,AQI_rolling7,AQI_change
0,2015-02-05,149.0,149.000000,0.000000
1,2015-02-06,190.0,169.500000,20.500000
2,2015-02-07,247.0,195.333333,51.666667
3,2015-02-08,379.0,241.250000,137.750000
4,2015-02-09,341.0,261.200000,79.800000
5,2015-02-10,256.0,260.333333,-4.333333
6,2015-02-11,388.0,278.571429,109.428571
7,2015-02-12,288.0,298.428571,-10.428571
8,2015-02-13,510.0,344.142857,165.857143
9,2015-02-14,761.0,417.571429,343.428571


In [151]:
df2['AQI_change_next'] = (
    df2.groupby('City')['AQI_change'].shift(-1)
)


In [152]:
df2 = df2.dropna(subset=['AQI_change_next'])


In [153]:
df2.head()

,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change,AQI_change_next
0,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54,2015-02-05,29.1,16.7,44.0,0.0,15.5,149.000000,0.000000,20.500000
1,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30,2015-02-06,29.9,16.7,52.0,0.0,11.0,169.500000,20.500000,51.666667
2,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76,2015-02-07,29.5,16.2,37.0,0.0,10.7,195.333333,51.666667,137.750000
3,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42,2015-02-08,28.7,14.6,37.0,0.0,10.7,241.250000,137.750000,79.800000
4,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94,2015-02-09,29.0,14.7,46.0,0.0,13.7,261.200000,79.800000,-4.333333


In [154]:
for lag in [1, 3, 7]:
    df2[f'AQI_lag_{lag}'] = (
        df2.groupby('City')['AQI'].shift(lag)
    )


In [155]:
for lag in [1, 3]:
    df2[f'PM10_lag_{lag}'] = (
        df2.groupby('City')['PM10'].shift(lag)
    )


In [156]:
df2.head()

,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3,...,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change,AQI_change_next,AQI_lag_1,AQI_lag_3,AQI_lag_7,PM10_lag_1,PM10_lag_3
0,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54,...,0.0,15.5,149.000000,0.000000,20.500000,NaN,NaN,NaN,NaN,NaN
1,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30,...,0.0,11.0,169.500000,20.500000,51.666667,149.0,NaN,NaN,122.41,NaN
2,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76,...,0.0,10.7,195.333333,51.666667,137.750000,190.0,NaN,NaN,122.41,NaN
3,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42,...,0.0,10.7,241.250000,137.750000,79.800000,247.0,149.0,NaN,122.41,122.41
4,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94,...,0.0,13.7,261.200000,79.800000,-4.333333,379.0,190.0,NaN,122.41,122.41


In [157]:
df2_sim = df2.copy()
df2_sim = df2_sim.sort_values(['City', 'Date']).reset_index(drop=True)

In [158]:
df2_sim['construction_intensity'] = 0.0

df2_sim['month'] = pd.to_datetime(df2_sim['Date']).dt.month
df2_sim['dow'] = pd.to_datetime(df2_sim['Date']).dt.dayofweek


In [159]:
#This code chooses a random value of construction intensity

import numpy as np

for city in df2_sim['City'].unique():

    city_mask = df2_sim['City'] == city
    city_dates = pd.to_datetime(df2_sim.loc[city_mask, 'Date'])

    for year in city_dates.dt.year.unique():

        year_mask = city_mask & (city_dates.dt.year == year)
        days = df2_sim.loc[year_mask, 'Date'].values

        if len(days) < 120:
            continue

        n_phases = np.random.choice([1,2,3], p=[0.2,0.5,0.3])

        for _ in range(n_phases):

            max_start = len(days) - 60
            if max_start <= 0:
                continue

            start_idx = np.random.randint(0, max_start)
            phase_len = np.random.randint(60, min(121, len(days)-start_idx))

            phase_dates = days[start_idx : start_idx + phase_len]

            mask = (df2_sim['Date'].isin(phase_dates)) & city_mask

            df2_sim.loc[mask, 'construction_intensity'] = \
                np.random.uniform(0.6, 1.0, size=mask.sum())


In [160]:
# Season boost (dry months higher)
df2_sim.loc[df2_sim['month'].isin([10,11,12,1,2,3,4,5]),
           'construction_intensity'] *= 1.15

# Monsoon suppression
df2_sim.loc[df2_sim['month'].isin([6,7,8,9]),
           'construction_intensity'] *= 0.7

# Weekend suppression
df2_sim.loc[df2_sim['dow'] >= 5,
           'construction_intensity'] *= 0.6

df2_sim['construction_intensity'] = df2_sim['construction_intensity'].clip(0,1)


In [161]:
df2_sim['construction_intensity'].mean()


np.float64(0.2552795867644515)

In [162]:
#In this block we add weights to the values, that would get affected the most due to construction
df2_sim['PM10'] = df2_sim['PM10'] * (1 + 0.45 * df2_sim['construction_intensity'])
df2_sim['PM2_5'] = df2_sim['PM2_5'] * (1 + 0.25 * df2_sim['construction_intensity'])
df2_sim['NO2'] = df2_sim['NO2'] * (1 + 0.07 * df2_sim['construction_intensity'])


**Feature Engineering**

In [163]:
df2_sim['construction_PM10'] = df2_sim['construction_intensity'] * df2_sim['PM10']
df2_sim['construction_PM25'] = df2_sim['construction_intensity'] * df2_sim['PM2_5']


In [164]:
df2_sim['AQI'] = (
    0.4 * df2_sim['PM2_5'] +
    0.3 * df2_sim['PM10'] +
    0.1 * df2_sim['NO2'] +
    0.1 * df2_sim['SO2'] +
    0.1 * df2_sim['O3']
)


In [165]:
df2_sim['construction_intensity'].describe()

,construction_intensity
count,21106.000000
mean,0.255280
std,0.348834
min,0.000000
25%,0.000000
50%,0.000000
75%,0.548628
max,1.000000


In [166]:
df2_sim['AQI_change_next'] = \
    df2_sim.groupby('City')['AQI'].shift(-1) - df2_sim['AQI']


In [167]:
df2_sim = df2_sim.dropna()


In [168]:
df2_sim['construction_rolling_7'] = (
    df2_sim.groupby('City')['construction_intensity']
          .rolling(7)
          .mean()
          .reset_index(level=0, drop=True)
)

df2_sim['construction_PM10'] = df2_sim['construction_intensity'] * df2_sim['PM10']


In [169]:
features = [
    'PM10','PM2_5','NO2','NO','SO2','CO','O3',
    'construction_intensity',
    'temperature_2m_max','temperature_2m_min',
    'relative_humidity_2m_mean','precipitation_sum',
    'wind_speed_10m_max'
]


X = df2_sim[features]
y = df2_sim['AQI_change_next']


In [170]:
df_feat = df2.copy()

# Ensure correct types
df_feat["Date"] = pd.to_datetime(df_feat["Date"])
df_feat = df_feat.sort_values(["City", "Date"])

# --- Lag features (AQI memory) ---
# Lag features help prediction because they store AQI values from previous days.

for lag in [1, 3, 7]:
    df_feat[f"AQI_lag_{lag}"] = df_feat.groupby("City")["AQI"].shift(lag)

# --- Rolling features (trend/volatility) ---
df_feat["AQI_roll7_mean"] = (
    df_feat.groupby("City")["AQI"]
    .rolling(window=7, min_periods=7)
    .mean()
    .reset_index(level=0, drop=True)
)

df_feat["AQI_roll7_std"] = (
    df_feat.groupby("City")["AQI"]
    .rolling(window=7, min_periods=7)
    .std()
    .reset_index(level=0, drop=True)
)

# Optional: rolling change (momentum)
df_feat["AQI_change_1d"] = df_feat.groupby("City")["AQI"].diff(1)
df_feat["AQI_change_3d"] = df_feat.groupby("City")["AQI"].diff(3)


In [171]:
np.random.seed(42)

if "construction_intensity" not in df_feat.columns:
    df_feat["construction_intensity"] = np.random.beta(2, 5, len(df_feat)) #Here we use beta distribution to simulte the values.

In [172]:
temporal_cols = [
    "AQI_lag_1","AQI_lag_3","AQI_lag_7",
    "AQI_roll7_mean","AQI_roll7_std",
    "AQI_change_1d","AQI_change_3d"
]

df_feat = df_feat.dropna(subset=temporal_cols).copy()
print("After temporal features, rows:", len(df_feat))

After temporal features, rows: 20952


In [173]:
features_updated = features + temporal_cols

In [174]:
df_sim = df_feat.sort_values(['City','Date'])

for lag in [1, 3, 7]:
    df_sim[f'AQI_lag_{lag}'] = df_sim.groupby('City')['AQI'].shift(lag)

df_sim['AQI_roll7_mean'] = df_sim.groupby('City')['AQI'].rolling(7).mean().reset_index(level=0, drop=True)
df_sim['AQI_roll7_std'] = df_sim.groupby('City')['AQI'].rolling(7).std().reset_index(level=0, drop=True)

In [175]:
print("Has construction_intensity?", "construction_intensity" in df_feat.columns)
print(df_feat.columns)


Has construction_intensity? True
Index(['City', 'Date', 'AQI', 'PM10', 'PM2_5', 'NO2', 'NO', 'SO2', 'CO', 'O3',
       'date', 'temperature_2m_max', 'temperature_2m_min',
       'relative_humidity_2m_mean', 'precipitation_sum', 'wind_speed_10m_max',
       'AQI_rolling7', 'AQI_change', 'AQI_change_next', 'AQI_lag_1',
       'AQI_lag_3', 'AQI_lag_7', 'PM10_lag_1', 'PM10_lag_3', 'AQI_roll7_mean',
       'AQI_roll7_std', 'AQI_change_1d', 'AQI_change_3d',
       'construction_intensity'],
      dtype='object')


In [176]:
target = "AQI_change_next"

model_df = df_sim[["City", "Date"] + features_updated + [target]].copy()

# Drop rows where ANY feature/target is missing
model_df = model_df.dropna(subset=features_updated + [target]).reset_index(drop=True)

print("Model df shape:", model_df.shape)


Model df shape: (20798, 23)


In [177]:
print(df_sim.columns)

Index(['City', 'Date', 'AQI', 'PM10', 'PM2_5', 'NO2', 'NO', 'SO2', 'CO', 'O3',
       'date', 'temperature_2m_max', 'temperature_2m_min',
       'relative_humidity_2m_mean', 'precipitation_sum', 'wind_speed_10m_max',
       'AQI_rolling7', 'AQI_change', 'AQI_change_next', 'AQI_lag_1',
       'AQI_lag_3', 'AQI_lag_7', 'PM10_lag_1', 'PM10_lag_3', 'AQI_roll7_mean',
       'AQI_roll7_std', 'AQI_change_1d', 'AQI_change_3d',
       'construction_intensity'],
      dtype='object')


In [178]:
backend_df = df_sim[[
    "City",
    "Date",
    "AQI",
    "PM10",
    "PM2_5",
    "NO2",
    "NO",
    "SO2",
    "CO",
    "O3"
]].copy()

backend_df = backend_df.sort_values(["City", "Date"])
backend_df.head()


,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3
7,Ahmedabad,2015-02-12,288.0,122.41,65.04,30.10,10.05,65.91,14.19,31.88
8,Ahmedabad,2015-02-13,510.0,122.41,103.36,39.56,10.05,80.43,18.18,40.11
9,Ahmedabad,2015-02-14,761.0,122.41,177.33,47.58,10.05,99.72,37.49,36.47
10,Ahmedabad,2015-02-15,475.0,122.41,113.25,30.50,10.05,69.71,15.55,33.78
11,Ahmedabad,2015-02-16,536.0,122.41,99.70,28.10,10.05,73.23,19.85,30.57


In [179]:
backend_df.to_csv("city_day.csv", index=False)

In [180]:
from sklearn.model_selection import train_test_split
model_df = model_df.sort_values(["City", "Date"]).reset_index(drop=True)

split_index = int(len(model_df) * 0.8)
train = model_df.iloc[:split_index]
test  = model_df.iloc[split_index:]

X_train = train[features_updated]
y_train = train[target]

X_test = test[features_updated]
y_test = test[target]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)


X_train: (16638, 20) y_train: (16638,)
X_test: (4160, 20) y_test: (4160,)


In [181]:
import lightgbm as lgb

reg = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42
)

reg.fit(X_train, y_train)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004873 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4905
[LightGBM] [Info] Number of data points in the train set: 16638, number of used features: 20
[LightGBM] [Info] Start training from score -0.214191


LGBMRegressor(learning_rate=0.05, n_estimators=300, random_state=42)

In [182]:
# ─── MODEL A: WEATHER-ONLY BASELINE ───────────────────────────
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
weather_features = [
    'temperature_2m_max', 'temperature_2m_min',
    'relative_humidity_2m_mean', 'precipitation_sum',
    'wind_speed_10m_max',
    # temporal AQI lags — these are fair game for both models
    'AQI_lag_1', 'AQI_lag_3', 'AQI_lag_7',
    'AQI_roll7_mean', 'AQI_roll7_std',
    'AQI_change_1d', 'AQI_change_3d'
]

X_train_A = train[weather_features]
X_test_A  = test[weather_features]

reg_A = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42
)
reg_A.fit(X_train_A, y_train)
y_pred_A = reg_A.predict(X_test_A)

rmse_A = np.sqrt(mean_squared_error(y_test, y_pred_A))
mae_A  = mean_absolute_error(y_test, y_pred_A)
r2_A   = r2_score(y_test, y_pred_A)

print("=== MODEL A (Weather Only) ===")
print(f"RMSE: {rmse_A:.4f}")
print(f"MAE:  {mae_A:.4f}")
print(f"R²:   {r2_A:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002188 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2865
[LightGBM] [Info] Number of data points in the train set: 16638, number of used features: 12
[LightGBM] [Info] Start training from score -0.214191
=== MODEL A (Weather Only) ===
RMSE: 32.3500
MAE:  20.9348
R²:   0.3812


In [183]:
y_pred = reg.predict(X_test)

In [184]:
pred_df = pd.DataFrame({
    "Actual_AQI_Change": y_test.values,
    "Predicted_AQI_Change": y_pred
})

pred_df.head(10)


,Actual_AQI_Change,Predicted_AQI_Change
0,-118.000000,-107.913432
1,-86.857143,-52.088395
2,-35.857143,-31.472256
3,38.142857,13.900597
4,51.428571,54.977721
5,99.571429,43.887476
6,36.142857,69.216018
7,-119.000000,-28.182619
8,-7.428571,-98.404853
9,81.857143,6.956672


In [185]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)


RMSE: 30.398121318224355
MAE: 20.338117032583128
R2: 0.45357647467056006


In [186]:
sample_df = X_test.iloc[[0]]

print("Input features:")
print(sample_df)

pred_one = reg.predict(sample_df)[0]

print("\nPredicted AQI Change:", pred_one)


Input features:
         PM10  PM2_5    NO2     NO  SO2    CO     O3  construction_intensity  \
16638  108.58  91.92  34.94  27.44  7.8  1.06  21.14                 0.39442   

       temperature_2m_max  temperature_2m_min  relative_humidity_2m_mean  \
16638                20.6                10.6                       79.0   

       precipitation_sum  wind_speed_10m_max  AQI_lag_1  AQI_lag_3  AQI_lag_7  \
16638                1.1                12.2      383.0      374.0      403.0   

       AQI_roll7_mean  AQI_roll7_std  AQI_change_1d  AQI_change_3d  
16638      381.857143      65.821548         -137.0         -128.0  

Predicted AQI Change: -107.91343174193845


In [187]:
custom_dict = {
    # pollutants
    "PM10": 150,
    "PM2_5": 80,
    "NO2": 40,
    "NO": 30,
    "SO2": 20,
    "CO": 1.2,
    "O3": 35,

    # construction + weather
    "construction_intensity": 0.5,
    "temperature_2m_max": 32,
    "temperature_2m_min": 20,
    "relative_humidity_2m_mean": 65,
    "precipitation_sum": 0.0,
    "wind_speed_10m_max": 10,

    # temporal AQI features (must be provided!)
    "AQI_lag_1": 180,
    "AQI_lag_3": 165,
    "AQI_lag_7": 150,
    "AQI_roll7_mean": 158,
    "AQI_roll7_std": 22,
    "AQI_change_1d": 15,
    "AQI_change_3d": 30
}

custom_df = pd.DataFrame([custom_dict])[features_updated]
pred_custom = reg.predict(custom_df)[0]
print("Predicted AQI Change:", pred_custom)


Predicted AQI Change: 28.6783883013648


In [188]:
threshold = train["AQI_change_next"].quantile(0.90)

model_df["spike"] = (model_df["AQI_change_next"] > threshold).astype(int)


In [189]:
split_index = int(len(model_df) * 0.8)
train = model_df.iloc[:split_index]
test  = model_df.iloc[split_index:]

# Regression targets (continuous)
X_train = train[features_updated]
y_train = train[target]  # e.g. "aqi_change"

X_test = test[features_updated]
y_test = test[target]

# Classification targets (binary) — define spike threshold
SPIKE_THRESHOLD = 20  # adjust based on your AQI scale

X_traincl = train[features_updated]
y_traincl = (train[target] > SPIKE_THRESHOLD).astype(int)

X_testcl = test[features_updated]
y_testcl = (test[target] > SPIKE_THRESHOLD).astype(int)

print("Class distribution:\n", y_traincl.value_counts())

Class distribution:
 AQI_change_next
0    12594
1     4044
Name: count, dtype: int64


In [190]:
from lightgbm import LGBMClassifier

clf = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    class_weight="balanced",   # VERY IMPORTANT
    random_state=42
)

clf.fit(X_traincl, y_traincl)

y_prob = clf.predict_proba(X_test)[:, 1]


[LightGBM] [Info] Number of positive: 4044, number of negative: 12594
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001137 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4905
[LightGBM] [Info] Number of data points in the train set: 16638, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [191]:
from sklearn.metrics import classification_report
y_pred = (y_prob > 0.3).astype(int)
print(classification_report(y_testcl, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.942     0.698     0.802      3276
           1      0.429     0.839     0.568       884

    accuracy                          0.728      4160
   macro avg      0.685     0.769     0.685      4160
weighted avg      0.833     0.728     0.752      4160



In [192]:
# ─── MODEL A: CLASSIFIER BASELINE ─────────────────────────────

clf_A = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=42
)
clf_A.fit(X_traincl[weather_features], y_traincl)

y_prob_A  = clf_A.predict_proba(X_testcl[weather_features])[:, 1]
y_pred_A_cls = (y_prob_A > 0.3).astype(int)

print("=== MODEL A CLASSIFIER (Weather Only) ===")
print(classification_report(y_testcl, y_pred_A_cls, digits=3))

[LightGBM] [Info] Number of positive: 4044, number of negative: 12594
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000683 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2865
[LightGBM] [Info] Number of data points in the train set: 16638, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
=== MODEL A CLASSIFIER (Weather Only) ===
              precision    recall  f1-score   support

           0      0.935     0.668     0.779      3276
           1      0.402     0.827     0.541       884

    accuracy                          0.702      4160
   macro avg      0.668     0.748     0.660      4160
weighted avg      0.821     0.702     0.729      4160



In [193]:
threshold = df_feat['AQI_change_next'].quantile(0.90)

df_feat['spike'] = (
    df_feat['AQI_change_next'] > threshold
).astype(int)
df_feat['spike'].value_counts(normalize=True)

,proportion
spike,
0,0.900105
1,0.099895


In [194]:
from lightgbm import LGBMClassifier

clf = LGBMClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)

clf.fit(X_train, y_traincl)

y_pred_class = clf.predict(X_test)


[LightGBM] [Info] Number of positive: 4044, number of negative: 12594
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006208 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4905
[LightGBM] [Info] Number of data points in the train set: 16638, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [195]:
# ─── HEAD TO HEAD COMPARISON ──────────────────────────────────

print("=" * 45)
print(f"{'Metric':<12} {'Model A (Weather)':>18} {'Model B (+ Constr)':>18}")
print("=" * 45)
print(f"{'RMSE':<12} {rmse_A:>18.4f} {rmse:>18.4f}")
print(f"{'MAE':<12} {mae_A:>18.4f} {mae:>18.4f}")
print(f"{'R²':<12} {r2_A:>18.4f} {r2:>18.4f}")
print("=" * 45)
print("Lower RMSE/MAE = better  |  Higher R² = better")
print("If Model B wins → H2 supported")
print("If Model A ties → construction proxy adds no signal")

Metric        Model A (Weather) Model B (+ Constr)
RMSE                    32.3500            30.3981
MAE                     20.9348            20.3381
R²                       0.3812             0.4536
Lower RMSE/MAE = better  |  Higher R² = better
If Model B wins → H2 supported
If Model A ties → construction proxy adds no signal


In [196]:
from sklearn.metrics import classification_report

print(classification_report(y_testcl, y_pred_class))


              precision    recall  f1-score   support

           0       0.90      0.86      0.88      3276
           1       0.56      0.66      0.60       884

    accuracy                           0.82      4160
   macro avg       0.73      0.76      0.74      4160
weighted avg       0.83      0.82      0.82      4160



In [197]:
print("features_updated length:", len(features_updated))
print(features_updated)


features_updated length: 20
['PM10', 'PM2_5', 'NO2', 'NO', 'SO2', 'CO', 'O3', 'construction_intensity', 'temperature_2m_max', 'temperature_2m_min', 'relative_humidity_2m_mean', 'precipitation_sum', 'wind_speed_10m_max', 'AQI_lag_1', 'AQI_lag_3', 'AQI_lag_7', 'AQI_roll7_mean', 'AQI_roll7_std', 'AQI_change_1d', 'AQI_change_3d']


In [198]:
print("features_updated:",
      len(features_updated))
print("reg n_features_:", reg.n_features_in_)
print("clf n_features_:", clf.n_features_in_)

features_updated: 20
reg n_features_: 20
clf n_features_: 20


In [199]:
import joblib

bundle = {
    "reg_model": reg,
    "clf_model": clf,
    "features": features_updated,
    "spike_prob_threshold": 0.30
}

joblib.dump(bundle, "aqi_bundle.pkl")


['aqi_bundle.pkl']